## RAG Pipeline
### Data Ingestion to Vector DB Pipelines

In [23]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [24]:
### Read all the pdfs inside the directory
def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            ### Adding Source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            all_documents.extend(documents)
            print(f"Loaded{len(documents)}pages")
        except Exception as e:
            print(f"Error:{e}")
    print(f"\nTotal Documents Loaded:{len(all_documents)}")
    return all_documents
all_pdf_documents=process_all_pdfs("../data/pdf")


Processing:molecular_biology.pdf
Loaded24pages

Processing:distributed_systems.pdf
Loaded24pages

Processing:history_of_computing.pdf
Loaded24pages

Processing:financial_markets.pdf
Loaded24pages

Processing:climate_science.pdf
Loaded24pages

Total Documents Loaded:120


In [25]:
### Split documents into smaller chunks using RecursiveCharacterTextSplitter
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk")
        print(f"\nContent:{split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")
    return split_docs

In [26]:
chunks=split_documents(all_pdf_documents);

Split 120 documents into 414 chunks

Example chunk

Content:Introduction to Molecular Biology
A synthetic long-form document generated for RAG ingestion testing.
The Structure of DNA and RNA
The Structure of DNA and RNA: Part 1
Experts note that mutation can s...
Metadata:{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-17T19:11:57+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-17T19:11:57+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/molecular_biology.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1', 'source_file': 'molecular_biology.pdf', 'file_type': 'pdf'}


In [27]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document

In [28]:
class EmbeddingsManager:
    """Handles document embeddings generation using sentence transformer"""
    def __init__(self,model_name: str="all-MiniLM-L6-v2"):
        """Initialize the embedding Manager
        Args:
            model_name:Hugging face model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model from Hugging Face"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str])->np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts:List of text strings to embed
        Returns:
            numpy array of embeddingsnwith shape (len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for{len(texts)}text...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape{embeddings.shape}")
        return embeddings
    
    def get_sentecnce_embedding_dimension(self)->int:
        """Get embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()


embedding_manager=EmbeddingsManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8630.77it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/rp/c1f4d6_x3_gb6mlttc5qss3c0000gn/T/ipykernel_89316/949910584.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [29]:
import hashlib

class VectorStore:
    """Manages document embeddings in a chromadb vector store"""
    def __init__(self,collection_name: str="pdf_documents",persist_directory: str="../data/vector_store"):
        """Initialize the Vector Store
        Args:
            collection_name:Name of the chromadb collection
            persist_directory:Directory to persist the chromadb data
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the chromadb persistent client and collection"""
        os.makedirs(self.persist_directory,exist_ok=True)
        self.client=chromadb.PersistentClient(
            path=self.persist_directory,
            settings=Settings(anonymized_telemetry=False)
        )
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"hnsw:space":"cosine"}
        )
        print(f"Vector store initialized. Collection:{self.collection_name}")
        print(f"Existing documents in collection:{self.collection.count()}")

    @staticmethod
    def _make_id(doc: Document) -> str:
        """Deterministic id from source file, page and content, so re-ingesting the same document upserts instead of duplicating"""
        key=f"{doc.metadata.get('source_file','')}|{doc.metadata.get('page','')}|{doc.page_content}"
        return hashlib.sha256(key.encode("utf-8")).hexdigest()

    @staticmethod
    def _sanitize_metadata(metadata: Dict[str,Any]) -> Dict[str,Any]:
        """Chroma only accepts str/int/float/bool metadata values"""
        sanitized={}
        for k,v in metadata.items():
            if v is None:
                sanitized[k]=""
            elif isinstance(v,(str,int,float,bool)):
                sanitized[k]=v
            else:
                sanitized[k]=str(v)
        return sanitized

    def add_documents(self,documents: List[Document],embeddings: np.ndarray,batch_size: int=500):
        """Add documents and their embeddings to the vector store
        Args:
            documents:List of langchain Document objects
            embeddings:numpy array of embeddings corresponding to the documents
            batch_size:Maximum number of documents to upsert per chromadb call
        """
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        ids=[self._make_id(doc) for doc in documents]
        texts=[doc.page_content for doc in documents]
        metadatas=[self._sanitize_metadata(doc.metadata) for doc in documents]
        embeddings_list=embeddings.tolist()

        for start in range(0,len(documents),batch_size):
            end=start+batch_size
            self.collection.upsert(
                ids=ids[start:end],
                embeddings=embeddings_list[start:end],
                documents=texts[start:end],
                metadatas=metadatas[start:end]
            )
        print(f"Upserted{len(documents)}documents to vector store")

    def query(self,query_embedding: np.ndarray,n_results: int=5) -> Dict[str,Any]:
        """Query the vector store for the most similar documents
        Args:
            query_embedding:embedding of the query text
            n_results:number of results to return
        Returns:
            chromadb query results dictionary
        """
        results=self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )
        return results

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection:pdf_documents
Existing documents in collection:414


In [30]:
### Convert text into embeddings
texts=[doc.page_content for doc in chunks]

### Generate embeddings
embeddings=embedding_manager.generate_embeddings(texts)

### Store in VectorDB
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for414text...


Batches: 100%|██████████| 13/13 [00:00<00:00, 15.54it/s]


Generated embeddings with shape(414, 384)
Upserted414documents to vector store


### Retriever Pipeline From VectorStore

In [31]:
class RAGRetriever:
    """Handles query-based retrieval from vector store"""

    def __init__(self,vector_store: VectorStore,embedding_manager: EmbeddingsManager):
        """Initialize the retriever
        Args:
            vector_store:VectorStore instance to search over
            embedding_manager:EmbeddingsManager instance used to embed queries
        """
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query: str,top_k: int=5,score_threshold: float=0.0) -> List[Dict[str,Any]]:
        """Retrieve the most relevant chunks for a query
        Args:
            query:Natural language query string
            top_k:Number of results to return
            score_threshold:Minimum similarity score (0-1) required to keep a result
        Returns:
            List of dicts with content, metadata and similarity score, ordered by relevance
        """
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]
        results=self.vector_store.query(query_embedding,n_results=top_k)

        retrieved=[]
        documents=results.get("documents",[[]])[0]
        metadatas=results.get("metadatas",[[]])[0]
        distances=results.get("distances",[[]])[0]

        for doc,metadata,distance in zip(documents,metadatas,distances):
            ### chromadb cosine distance -> similarity score
            similarity=1-distance
            if similarity>=score_threshold:
                retrieved.append({
                    "content":doc,
                    "metadata":metadata,
                    "score":similarity
                })

        print(f"Retrieved {len(retrieved)} chunks for query:{query!r}")
        return retrieved


rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [32]:
rag_retriever.retrieve("where is climate science")

Generating embeddings for1text...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.85it/s]

Generated embeddings with shape(1, 384)
Retrieved 5 chunks for query:'where is climate science'


[{'content': 'Regional Climate Impacts\nRegional Climate Impacts: Part 1\nA central challenge in this area involves balancing emissions scenario against permafrost while accounting\nfor methane. Several models attempt to explain the relationship between carbon sink and extreme\nweather. Understanding thermohaline circulation requires careful analysis of emissions scenario as well as\nocean acidification. In practice, regional climate impacts depends heavily on the behavior of tipping point\nunder varying conditions. Researchers have long studied how extreme weather interacts with renewable\nenergy in the context of regional climate impacts. In practice, regional climate impacts depends heavily on\nthe behavior of albedo under varying conditions.\nHistorically, advances in regional climate impacts were driven by improvements in radiative forcing.\nUnderstanding carbon sink requires careful analysis of methane as well as carbon dioxide. Researchers',
  'metadata': {'moddate': '2026-09-17

### Integration VectorDB Context pipeline with output

In [33]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

Chat_openai_key=os.getenv("OPENAI_API_KEY")


In [34]:
llm=ChatOpenAI(api_key=Chat_openai_key,model_name="gpt-3.5-turbo",temperature=0.1)

def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context Found"

    prompt=f""" Use the following Context to answer the question consiely
    Context:{context}
    Question:{query}
    Answer:
    """

    response=llm.invoke([prompt])

    return response.content

In [36]:
answer=rag_simple("what is consensus Algorithm",rag_retriever,llm)
print(answer)

Generating embeddings for1text...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.34it/s]


Generated embeddings with shape(1, 384)
Retrieved 3 chunks for query:'what is consensus Algorithm'
Consensus Algorithm is a method used in distributed systems to achieve an agreement among multiple nodes on a single data value or decision. It involves balancing factors such as linearizability, partition, commit log, latency, failover, vector clock, replica, leader election, eventual consistency, quorum, sharding, and gossip protocol.


### Advace RAG Pipeline

In [38]:
def rag_advance(query,rag_retriever,llm,top_k=5,min_score=0.2,return_context=False):
    """ RAG pipeline with extra features
    -Return answer,sources,confidence score and optionally full context
    """
    results=rag_retriever.retrieve(query,top_k=top_k, score_threshold=min_score)

    if not results:
        return {'answer':'No relevant context found.','sources':[],'confidence':0.0,'context':''}
    context='\n\n'.join([doc['content']for doc in results])

    sources=[{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unkown')
    } for doc in results]
    confidence=max([doc['score']for doc in results])

    prompt=f"""Use the following context to answer the question concisely and accurately.
    If the context does not contain enough information to answer, say so instead of guessing.

    Context:{context}

    Question:{query}
    Answer:
    """

    response=llm.invoke([prompt])
    answer=response.content

    result={'answer':answer,'sources':sources,'confidence':confidence}
    if return_context:
        result['context']=context
    return result

In [39]:
result=rag_advance("what is consensus Algorithm",rag_retriever,llm)
print("Answer:",result['answer'])
print("Confidence:",result['confidence'])
print("Sources:",result['sources'])

Generating embeddings for1text...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.05it/s]


Generated embeddings with shape(1, 384)
Retrieved 5 chunks for query:'what is consensus Algorithm'
Answer: Consensus Algorithm is a process used in distributed systems to achieve an agreement on a single data value among multiple nodes.
Confidence: 0.6336047649383545
Sources: [{'source': 'distributed_systems.pdf', 'page': 6}, {'source': 'distributed_systems.pdf', 'page': 6}, {'source': 'distributed_systems.pdf', 'page': 7}, {'source': 'distributed_systems.pdf', 'page': 7}, {'source': 'distributed_systems.pdf', 'page': 6}]
